# Aula 8 — Backtest Rigoroso

Nas últimas aulas construímos duas versões de sinal e comparamos suas métricas. Os números parecem bons. Mas será que podemos confiar neles?

Esta aula responde essa pergunta. Vamos aprender a **identificar e corrigir** os principais problemas de um backtest.

---

**Agenda:**
1. Look-ahead bias — quando o futuro contamina o passado
2. Overfitting — a estratégia que só funciona no treino
3. Multiple testing — o problema das comparações múltiplas
4. Deflated Sharpe Ratio — ajustando pelo número de testes
5. Custos reais de transação — o que a maioria ignora
6. Walk-forward validation — a forma mais honesta de avaliar

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Carregando dados das aulas anteriores
ret_diarios  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'))
ret_mensais  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
ret_ibov     = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))['BOVA11.SA']
sinal_v1     = pd.read_parquet(os.path.join(DADOS_DIR, 'sinal_v1.parquet'))
sinal_v2     = pd.read_parquet(os.path.join(DADOS_DIR, 'sinal_v2.parquet'))
pesos_v1     = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_v1.parquet'))
pesos_v2     = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_sinal_v2.parquet'))
ret_v1       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet')).squeeze()
ret_v2       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_sinal_v2.parquet')).squeeze()

# Funções de métricas (copiadas para auto-contenção)
def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    r = retornos.dropna()
    excesso = r - rf_mensal
    n = len(r)
    cagr = (1 + r).prod() ** (12 / n) - 1
    vol  = r.std() * np.sqrt(12)
    sharpe = (excesso.mean() / excesso.std()) * np.sqrt(12) if excesso.std() > 0 else np.nan
    downside = excesso[excesso < 0].std() * np.sqrt(12)
    sortino  = (excesso.mean() * 12 / downside) if downside > 0 else np.nan
    cum = (1 + r).cumprod()
    dd  = cum / cum.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan
    alpha, beta = np.nan, np.nan
    if benchmark is not None:
        b = benchmark.reindex(r.index).dropna()
        r2 = r.reindex(b.index)
        if len(r2) > 10:
            beta, intercept, *_ = stats.linregress(b.values, r2.values)
            alpha = intercept * 12
    return dict(nome=nome, cagr=cagr, vol=vol, sharpe=sharpe, sortino=sortino,
                max_drawdown=mdd, calmar=calmar, alpha=alpha, beta=beta)

def exibir_metricas(*dicts_metricas):
    fmt = {'cagr': '{:.1%}', 'vol': '{:.1%}', 'sharpe': '{:.2f}',
           'sortino': '{:.2f}', 'max_drawdown': '{:.1%}',
           'calmar': '{:.2f}', 'alpha': '{:.1%}', 'beta': '{:.2f}'}
    labels = {'cagr': 'CAGR', 'vol': 'Vol Anual', 'sharpe': 'Sharpe',
              'sortino': 'Sortino', 'max_drawdown': 'Max Drawdown',
              'calmar': 'Calmar', 'alpha': 'Alpha a.a.', 'beta': 'Beta'}
    nomes = [d['nome'] for d in dicts_metricas]
    print(f"{'Métrica':<18}" + ''.join(f"{n:>14}" for n in nomes))
    print('-' * (18 + 14 * len(nomes)))
    for k, label in labels.items():
        vals = []
        for d in dicts_metricas:
            v = d.get(k, np.nan)
            vals.append(fmt[k].format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A')
        print(f"{label:<18}" + ''.join(f"{v:>14}" for v in vals))

print('Setup OK.')

## 1. O Erro mais Comum no Backtesting: Look-Ahead Bias

O **Look-Ahead Bias (Viés de Olhar à Frente)** é o pecado capital do modelador quantitativo. Ele ocorre quando utilizamos, de forma direta ou indireta, informações do futuro para tomar uma decisão de investimento no passado.

---

### Como Acontece no Código de Forma Sutil

A forma mais comum em que iniciantes caem nessa armadilha é na alocação de datas dos retornos e preços.
Imagine o seguinte código conceitual:
- No final do mês de Janeiro ($t$), você calcula o retorno médio das ações ao longo do mês de Janeiro.
- Você usa esses dados para decidir quais ações comprar no início do mês de Janeiro ($t$).

Parece um erro óbvio de "tolo"? No entanto, ele acontece de forma extremamente oculta quando manipulamos arrays de DataFrames sem a devida atenção de índices. Se você comprar ações no início de Janeiro com base nos retornos acumulados que **só seriam consolidados e conhecidos no final do próprio mês de Janeiro**, seu backtest registrará uma performance maravilhosa que seria **fisicamente impossível** de executar em tempo real na realidade.

---

### O Salvador Cósmico: A Função `.shift(1)` no Pandas

Para nos protegermos contra essa falha estrutural, usamos de forma rigorosa e sistemática a operação de **deslocamento temporal** no Pandas:

```python
# O peso da carteira para o período t deve ser baseado no sinal calculado até o período t-1
pesos_executados = pesos_calculados.shift(1)
```

Essa linha de código garante que o peso que você está aplicando para calcular os retornos das ações no mês de Fevereiro foi determinado **exclusivamente** com os dados fechados e conhecidos até o último dia útil de Janeiro. Nenhuma informação de Fevereiro foi usada na tomada de decisão. Na competição do Itaú Asset, um único deslize de look-ahead bias desclassifica a equipe instantaneamente.


In [ ]:
# Estratégia CONTAMINADA (usa retorno do mês corrente no sinal)
sinal_contaminado = ret_mensais.rolling(12).sum()  # sem shift
rank_contaminado  = sinal_contaminado.rank(axis=1, ascending=True, pct=True)
top10_contaminado = (rank_contaminado >= 0.9).astype(float)
soma_contaminado  = top10_contaminado.sum(axis=1).replace(0, np.nan)
pesos_contaminado = top10_contaminado.div(soma_contaminado, axis=0).dropna(how='all')
ret_contaminado   = (pesos_contaminado.shift(1) * ret_mensais).sum(axis=1)

# Estratégia correta (v1) — já calculada antes
# Aqui isolamos apenas o efeito do shift para comparação limpa
sinal_correto = ret_mensais.shift(2).rolling(11).sum()  # nosso sinal v1
rank_correto  = sinal_correto.rank(axis=1, ascending=True, pct=True)
top10_correto = (rank_correto >= 0.9).astype(float)
soma_correto  = top10_correto.sum(axis=1).replace(0, np.nan)
pesos_correto = top10_correto.div(soma_correto, axis=0).dropna(how='all')
ret_correto   = (pesos_correto.shift(1) * ret_mensais).sum(axis=1)

# Alinhar e calcular métricas
idx = ret_contaminado.dropna().index.intersection(ret_correto.dropna().index)
m_contaminado = calcular_metricas(ret_contaminado.loc[idx], benchmark=ret_ibov.reindex(idx),
                                   nome='CONTAMINADO (look-ahead)')
m_correto     = calcular_metricas(ret_correto.loc[idx], benchmark=ret_ibov.reindex(idx),
                                   nome='CORRETO (sem look-ahead)')

print('=== IMPACTO DO LOOK-AHEAD BIAS ===')
exibir_metricas(m_contaminado, m_correto)
print(f"\nInflação do Sharpe: {m_contaminado['sharpe']:.2f} → {m_correto['sharpe']:.2f}")
print(f"Isso representa {m_contaminado['sharpe']/m_correto['sharpe'] - 1:.0%} de inflação artificial")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Retorno cumulativo
ax = axes[0]
((1 + ret_contaminado.loc[idx]).cumprod() - 1).plot(ax=ax, color='red', lw=2, label='Contaminado (look-ahead)')
((1 + ret_correto.loc[idx]).cumprod() - 1).plot(ax=ax, color='steelblue', lw=2, label='Correto')
((1 + ret_ibov.reindex(idx)).cumprod() - 1).plot(ax=ax, color='gray', lw=1.5, linestyle='--', label='IBOVESPA')
ax.set_title('Retorno Cumulativo: impacto do look-ahead bias')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.legend()

# Diferença mês a mês
ax = axes[1]
diff = ret_contaminado.loc[idx] - ret_correto.loc[idx]
colors = ['red' if v < 0 else 'steelblue' for v in diff]
ax.bar(diff.index, diff.values, color=colors, alpha=0.7)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Diferença mensal: contaminado − correto')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))
ax.set_ylabel('Diferença de retorno')

plt.tight_layout()
plt.savefig('lookahead_bias.png', dpi=120, bbox_inches='tight')
plt.show()
print('A diferença positiva (contaminado > correto) é o "presente gratuito" do futuro.')

### Outros tipos de look-ahead bias

| Tipo | Como acontece | Proteção |
|------|---------------|----------|
| **Sinal** | Usar retorno do mês corrente no signal | `.shift()` correto |
| **Survivorship bias** | Backtesting apenas nas ações que sobreviveram | Universe dinâmico (ex: IBOV constituintes históricos) |
| **Point-in-time** | Usar dados revisados (ex: lucro ajustado) em vez dos divulgados | Dados point-in-time |
| **Target leakage** | Features de ML que contêm o target indiretamente | Revisão cuidadosa de features |

No nosso caso, usamos `yfinance` com preços ajustados de dividendos e splits — isso é OK para pesquisa, mas em produção precisaríamos de um provider point-in-time.

## 2. Overfitting (Sobreajuste): O Autoengano de Parâmetros Perfeitos

O **Overfitting** é o inimigo silencioso do cientista de dados e do quant financeiro. Ele ocorre quando criamos um modelo que descreve perfeitamente o ruído histórico dos dados passados, mas falha gravemente ao lidar com dados novos no futuro.

---

### A Anatomia do Overfitting Quantitativo

Como iniciantes em pesquisa quant, é natural pensarmos: *"E se eu testar todas as combinações possíveis de parâmetros para encontrar a melhor estratégia?"*
- Você testa a janela de momentum de 3 meses, 4 meses, 5 meses... até 24 meses.
- Você testa pular 1 mês, 2 meses ou nenhum.
- Você testa rebalanceamento a cada 5 dias, 10 dias ou 20 dias.
- Você testa filtros de liquidez de R$ 1M a R$ 50M de volume médio diário.

Depois de rodar milhares de simulações, você encontra **"A Estratégia Perfeita"**: Momentum de 11.5 meses, pulando 1.2 meses, com rebalanceamento a cada 22 dias.
- No backtest histórico, ela tem uma rentabilidade espetacular e Sharpe de 2.5!
- **A triste realidade:** Essa combinação de parâmetros é altamente específica para o ruído aleatório daquele histórico exato. Ao colocar essa estratégia para rodar com dinheiro real a partir de hoje (out-of-sample), ela performará de forma horrível, retornando à média histórica decepcionante. O modelo decorou o passado em vez de aprender um padrão econômico real.


In [ ]:
def backtest_window(ret_mensais, window, n_top=10):
    """Backtest de momentum cross-sectional para uma janela específica."""
    sinal = ret_mensais.shift(2).rolling(window).sum()
    rank  = sinal.rank(axis=1, ascending=True, pct=True)
    top   = (rank >= (1 - n_top / rank.shape[1])).astype(float)
    soma  = top.sum(axis=1).replace(0, np.nan)
    pesos = top.div(soma, axis=0)
    ret   = (pesos.shift(1) * ret_mensais).sum(axis=1).dropna()
    if len(ret) < 24:
        return np.nan
    excesso = ret
    return (excesso.mean() / excesso.std()) * np.sqrt(12)

# Testamos janelas de 2 a 36 meses (35 parâmetros)
windows = list(range(2, 37))
sharpes_is = []
for w in windows:
    s = backtest_window(ret_mensais, w)
    sharpes_is.append(s)

df_sweep = pd.DataFrame({'window': windows, 'sharpe_is': sharpes_is}).dropna()
melhor_window = df_sweep.loc[df_sweep['sharpe_is'].idxmax(), 'window']
melhor_sharpe = df_sweep['sharpe_is'].max()

print(f'Janelas testadas: {len(df_sweep)}')
print(f'Melhor janela in-sample: {melhor_window} meses')
print(f'Melhor Sharpe in-sample: {melhor_sharpe:.2f}')
print(f'Sharpe da nossa janela (11): {df_sweep[df_sweep["window"]==11]["sharpe_is"].values[0]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva de Sharpe vs janela
ax = axes[0]
ax.plot(df_sweep['window'], df_sweep['sharpe_is'], color='steelblue', lw=2, marker='o', ms=4)
ax.axvline(11, color='green', linestyle='--', lw=2, label='Nossa janela (11) — fundamentada')
ax.axvline(melhor_window, color='red', linestyle='--', lw=2, label=f'Melhor in-sample ({melhor_window}) — data-mined')
ax.axhline(df_sweep['sharpe_is'].mean(), color='gray', linestyle=':', label=f'Média ({df_sweep["sharpe_is"].mean():.2f})')
ax.set_xlabel('Janela de lookback (meses)')
ax.set_ylabel('Sharpe in-sample')
ax.set_title('Overfitting: Sharpe por janela')
ax.legend()

# Distribuição dos Sharpes (todos os parâmetros)
ax = axes[1]
ax.hist(df_sweep['sharpe_is'].dropna(), bins=12, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(melhor_sharpe, color='red', lw=2, linestyle='--', label=f'Máximo escolhido: {melhor_sharpe:.2f}')
ax.axvline(df_sweep[df_sweep['window']==11]['sharpe_is'].values[0], color='green', lw=2,
           linestyle='--', label='Janela fundamentada (11)')
ax.set_xlabel('Sharpe in-sample')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Sharpes por parâmetro')
ax.legend()

plt.tight_layout()
plt.savefig('overfitting_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nLição: a janela 'melhor' in-sample foi encontrada por busca.")
print("Nossa janela de 11 meses foi escolhida por fundamento econômico (Jegadeesh & Titman, skip month).")
print("Isso é crucial: fundamento > otimização.")

### Como evitar overfitting

1. **Fundamento econômico primeiro** — defina o parâmetro pela teoria, não pela otimização
2. **Menos parâmetros** — cada parâmetro livre é uma oportunidade de overfit
3. **Walk-forward validation** — teste sempre em dados que não foram vistos durante a otimização (seção 6)
4. **Deflated Sharpe** — penaliza o Sharpe pelo número de testes realizados (seção 4)

## 3. O Problema das Comparações Múltiplas (Data Snooping / P-Hacking)

Quando testamos várias hipóteses até achar uma que funcione, caímos num problema clássico de probabilidade chamado **Data Snooping** ou **Multiple Testing Bias**.

---

### A Intuição Estatística com Lançamento de Moedas
Se você der uma moeda justa para **1 pessoa** e pedir para ela lançar 10 vezes consecutivas, a probabilidade de ela obter 10 caras seguidas é extremamente baixa:
$$P(10 \text{ caras}) = 0.5^{10} \approx 0.097\% \quad (\text{quase impossível})$$
Se isso acontecer, você desconfiará legitimamente que a moeda é fraudada.

No entanto, se você colocar **10.000 pessoas** numa sala com moedas justas e pedir para todas lançarem a moeda 10 vezes:
- A probabilidade estatística de que **pelo menos uma pessoa** obtenha 10 caras seguidas na sala é de quase **99.9%** por pura sorte!
- Se você selecionar apenas essa pessoa vitoriosa e contratá-la como o "Mago Supremo da Moeda", você estará cometendo um erro catastrófico de seleção.

O mesmo ocorre no backtesting quantitativo. Se você rodar 1.000 estratégias com sinais puramente aleatórios (lixo de dados), a pura probabilidade garante que cerca de 50 delas exibirão Sharpe ratios elevados e p-valores estatisticamente significantes de forma espúria no histórico de treino. Sem correção para testes múltiplos, você contratará sorte achando que contratou habilidade.


In [ ]:
np.random.seed(42)
n_simulacoes = 1000
n_meses      = len(ret_v1.dropna())

sharpes_aleatorios = []
for _ in range(n_simulacoes):
    # Estratégia completamente aleatória: pesos uniformes sobre ações aleatórias
    ret_rand = np.random.choice(ret_v1.dropna().values, size=n_meses, replace=True)
    s = (np.mean(ret_rand) / np.std(ret_rand)) * np.sqrt(12)
    sharpes_aleatorios.append(s)

sharpes_aleatorios = np.array(sharpes_aleatorios)
sharpe_real_v2 = calcular_metricas(ret_v2, nome='v2')['sharpe']

# Quantas estratégias aleatórias batem o nosso Sharpe?
pct_bateu = (sharpes_aleatorios > sharpe_real_v2).mean()
p_valor_empirico = pct_bateu

print(f'Sharpe da estratégia v2: {sharpe_real_v2:.2f}')
print(f'Sharpe médio de estratégias aleatórias: {sharpes_aleatorios.mean():.2f}')
print(f'Estratégias aleatórias que bateram v2: {pct_bateu:.1%}')
print(f'p-valor empírico: {p_valor_empirico:.4f}')

# Quantas veríamos por acaso se testássemos N estratégias com threshold 5%?
print('\nFalsos positivos esperados por número de testes (threshold = 5%):')
for n_testes in [1, 10, 20, 50, 100]:
    falsos = 1 - (1 - 0.05) ** n_testes
    print(f'  {n_testes:3d} testes → probabilidade de pelo menos 1 falso positivo: {falsos:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(sharpes_aleatorios, bins=50, color='lightgray', edgecolor='white', label='Estratégias aleatórias (n=1000)')
ax.axvline(sharpe_real_v2, color='green', lw=2.5, label=f'Nossa estratégia v2 (Sharpe={sharpe_real_v2:.2f})')
ax.axvline(np.percentile(sharpes_aleatorios, 95), color='orange', lw=2, linestyle='--',
           label=f'Percentil 95% do ruído ({np.percentile(sharpes_aleatorios, 95):.2f})')

ax.set_xlabel('Sharpe Ratio')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Sharpe sob hipótese nula (puro ruído)')
ax.legend()

plt.tight_layout()
plt.savefig('multiple_testing.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Deflated Sharpe Ratio (DSR) — A Solução de Marcos López de Prado

Para combater formalmente o viés de comparações múltiplas no processo quantitativo, o renomado pesquisador Marcos López de Prado propôs o **DSR (Deflated Sharpe Ratio)**.

### A Filosofia por Trás do DSR
O DSR é uma formulação matemática rigorosa que calcula a probabilidade de o Sharpe Ratio de uma estratégia ser estatisticamente superior a zero, **descontando o número de tentativas (testes) feitas até encontrar o resultado final**.

O DSR ajusta o Sharpe estimado com base em quatro parâmetros críticos:
1. **$SR_0$ (Sharpe Esperado sob Hipótese Nula):** Geralmente zero.
2. **$N$ (Número de Estratégias Testadas):** Quantas simulações você rodou no total antes de escolher a vencedora.
3. **$\text{Var}(\{SR\})$ (Variância dos Sharpes obtidos):** O quão diferentes foram as estratégias testadas entre si.
4. **$T$ (Tamanho do Histórico):** A duração dos dados em número de observações.

O DSR atua como um teste de hipóteses ultra rigoroso: se você realizou centenas de tentativas para achar um Sharpe de 1.2, o DSR "deflacionará" a significância estatística desse Sharpe, mostrando se ele de fato é real ou apenas o resultado de um *p-hacking* intensivo. Utilizar o DSR no seu projeto demonstrará um nível de maturidade metodológica espetacular para a banca examinadora do Itaú.


In [ ]:
from scipy.stats import norm

def sharpe_minimo_necessario(n_testes, n_obs, sharpe_std=1.0):
    """
    Sharpe mínimo necessário para ser estatisticamente significativo
    dado que foram realizados n_testes testes com n_obs observações cada.
    
    Bailey & López de Prado (2014): Deflated Sharpe Ratio.
    Retorna o benchmark de Sharpe ajustado pelo número de testes.
    """
    EULER = 0.5772156649
    
    if n_testes == 1:
        return norm.ppf(0.95) * sharpe_std / np.sqrt(n_obs)
    
    # Valor esperado do máximo de n_testes normais (aprox.)
    sr_benchmark = sharpe_std * (
        (1 - EULER) * norm.ppf(1 - 1/n_testes) +
        EULER * norm.ppf(1 - 1/(n_testes * np.e))
    )
    return sr_benchmark

n_obs = len(ret_v2.dropna())

print(f'Observações disponíveis: {n_obs} meses')
print(f'Sharpe observado (v2): {sharpe_real_v2:.2f}')
print()
print(f"{'N testes':<12} {'Sharpe mínimo':>15} {'Nossa estratégia passa?':>25}")
print('-' * 54)
for n in [1, 5, 10, 20, 35, 50, 100]:
    sr_min = sharpe_minimo_necessario(n, n_obs)
    passa = 'SIM ✓' if sharpe_real_v2 > sr_min else 'NÃO ✗'
    print(f"{n:<12} {sr_min:>15.2f} {passa:>25}")

In [ ]:
ns_testes = list(range(1, 101))
benchmarks = [sharpe_minimo_necessario(n, n_obs) for n in ns_testes]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns_testes, benchmarks, color='orange', lw=2, label='Sharpe mínimo necessário')
ax.axhline(sharpe_real_v2, color='green', lw=2, linestyle='--', label=f'Nossa estratégia v2 ({sharpe_real_v2:.2f})')
ax.fill_between(ns_testes, benchmarks, sharpe_real_v2,
                where=[b < sharpe_real_v2 for b in benchmarks],
                alpha=0.15, color='green', label='Zona de aprovação')
ax.fill_between(ns_testes, benchmarks, sharpe_real_v2,
                where=[b >= sharpe_real_v2 for b in benchmarks],
                alpha=0.15, color='red', label='Zona de rejeição')

# Marcar quantos testes fizemos de fato (35 janelas)
ax.axvline(35, color='gray', lw=1.5, linestyle=':', label='Testes realizados (sweep = 35)')

ax.set_xlabel('Número de estratégias/parâmetros testados')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Deflated Sharpe: barreira sobe com o número de testes')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('deflated_sharpe.png', dpi=120, bbox_inches='tight')
plt.show()

sr_min_35 = sharpe_minimo_necessario(35, n_obs)
print(f'\nCom 35 testes (nosso sweep de janelas), o Sharpe mínimo é: {sr_min_35:.2f}')
print(f'Nossa estratégia v2: {sharpe_real_v2:.2f} → {"APROVADA" if sharpe_real_v2 > sr_min_35 else "REPROVADA"}')
print()
print('Mas atenção: não fizemos 35 testes de verdade.')
print('Escolhemos a janela 11 por fundamento econômico, antes de ver os dados.')
print('Isso muda completamente a interpretação estatística.')

### O valor do fundamento econômico

Quando a escolha de parâmetros é **baseada em teoria** (Jegadeesh & Titman 1993 → janela 12-1), o número efetivo de testes é próximo de **1**. Isso é muito diferente de testar 35 janelas e escolher a melhor.

No relatório da competição, você deve **justificar cada escolha metodológica** com fundamento da literatura. Isso não é só acadêmico — é o que separa uma estratégia robusta de uma estratégia data-mined.

## 5. Custos Reais de Transação: O Choque de Realidade do Backtest Líquido

No backtest conceitual (teórico), assumimos que o mercado é ideal: você pode comprar e vender ações no valor exato do preço de fechamento, com taxas zero. Na vida real de uma mesa quantitativa do Itaú Asset, **os custos operacionais devoram grande parte do alfa se a estratégia não for otimizada**.

---

### Os Três Elementos do Custo Real de Transação

1. **Taxas Explícitas da Bolsa (B3):**
   - Emolumentos e taxas de liquidação cobrados pela B3 por cada compra ou venda (aproximadamente $0.03\%$ por transação).
   - Taxas de corretagem explícitas cobradas pelas corretoras.

2. **Custo de Aluguel de Ações (para estratégias short):**
   - Para vender uma ação a descoberto, você precisa alugar esse ativo de outro acionista, pagando uma taxa anualizada de aluguel (que varia de $0.5\%$ a mais de $10\%$ ao ano, dependendo da escassez do papel no mercado).

3. **Slippage e Impacto de Mercado (O Custo Implícito mais Severo):**
   - Se você precisa gerenciar um fundo quantitativo de R$ 50 Milhões e decide comprar R$ 5 Milhões da ação de média liquidez no início do mês:
   - Você não conseguirá comprar tudo no último preço de tela. Suas ordens sucessivas de compra consumirão as ofertas do livro, empurrando o preço médio de execução para cima.
   - Essa perda de eficiência em relação ao preço teórico ideal de fechamento é chamada de **Slippage**.

### A Modelagem em Backtests
Para tornar nosso backtest honesto e digno de produção real, aplicamos um desconto de taxa fixa (por exemplo, de $0.1\%$ a $0.3\%$) sobre o volume financeiro girado a cada rebalanceamento (Turnover). Você verá no código a diferença gritante entre a curva de capital bruta (sem custos) e a curva líquida operacional real.


In [ ]:
# Calcular turnover da estratégia v2
turnover_v2 = pesos_v2.diff().abs().sum(axis=1) / 2
turnover_v2 = turnover_v2.dropna()

print(f'Turnover médio mensal (v2): {turnover_v2.mean():.1%}')
print(f'Turnover mediano mensal (v2): {turnover_v2.median():.1%}')
print()

# Cenários de custo
custos = {
    'Sem custo (naive)': 0.000,
    'Custo baixo (0.3%)': 0.003,
    'Custo médio (0.5%)': 0.005,
    'Custo alto (1.0%)': 0.010,
}

resultados_custo = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1, ax2 = axes
cores = ['steelblue', 'green', 'orange', 'red']

for (nome_custo, custo_por_turno), cor in zip(custos.items(), cores):
    # Subtrair custo proporcional ao turnover
    custo_mensal = (turnover_v2 * custo_por_turno).reindex(ret_v2.index)
    ret_liquido  = ret_v2 - custo_mensal.fillna(0)
    
    m = calcular_metricas(ret_liquido.dropna(), benchmark=ret_ibov, nome=nome_custo)
    resultados_custo.append(m)
    
    ((1 + ret_liquido.dropna()).cumprod() - 1).plot(ax=ax1, label=nome_custo, color=cor, lw=1.8)

((1 + ret_ibov).cumprod() - 1).plot(ax=ax1, label='IBOVESPA', color='gray', lw=1.5, linestyle='--')
ax1.set_title('Retorno cumulativo por cenário de custo')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax1.legend(fontsize=9)

# Degradação do Sharpe
sharpes = [m['sharpe'] for m in resultados_custo]
nomes_curtos = ['0%', '0.3%', '0.5%', '1.0%']
bars = ax2.bar(nomes_curtos, sharpes, color=cores, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, sharpes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', fontweight='bold')
ax2.set_xlabel('Custo por turno')
ax2.set_ylabel('Sharpe Ratio')
ax2.set_title('Degradação do Sharpe com custos de transação')
ax2.axhline(1.0, color='gray', linestyle=':', alpha=0.7, label='Sharpe = 1.0')
ax2.legend()

plt.tight_layout()
plt.savefig('custos_transacao.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== IMPACTO DOS CUSTOS ===')
exibir_metricas(*resultados_custo)

In [ ]:
# Ponto de indiferença: qual custo zera o alpha?
custos_granulares = np.linspace(0, 0.02, 200)
alphas_custo = []
for c in custos_granulares:
    custo_mensal = (turnover_v2 * c).reindex(ret_v2.index)
    ret_liq = ret_v2 - custo_mensal.fillna(0)
    m = calcular_metricas(ret_liq.dropna(), benchmark=ret_ibov, nome='')
    alphas_custo.append(m.get('alpha', np.nan))

alphas_custo = np.array(alphas_custo)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(custos_granulares * 100, alphas_custo * 100, color='steelblue', lw=2)
ax.axhline(0, color='red', lw=1.5, linestyle='--')
ax.fill_between(custos_granulares * 100, alphas_custo * 100, 0,
                where=alphas_custo > 0, alpha=0.15, color='green', label='Alpha positivo')
ax.fill_between(custos_granulares * 100, alphas_custo * 100, 0,
                where=alphas_custo <= 0, alpha=0.15, color='red', label='Alpha negativo')

# Break-even
if np.any(alphas_custo <= 0):
    break_even_idx = np.argmax(alphas_custo <= 0)
    break_even = custos_granulares[break_even_idx] * 100
    ax.axvline(break_even, color='orange', lw=2, linestyle='--',
               label=f'Break-even: {break_even:.2f}% por turno')

ax.axvline(0.5, color='gray', lw=1.5, linestyle=':', label='Estimativa B3 (~0.5%)')
ax.set_xlabel('Custo por turno (%)')
ax.set_ylabel('Alpha anual (%)')
ax.set_title('Break-even de custos: quando o alpha desaparece?')
ax.legend()

plt.tight_layout()
plt.savefig('breakeven_custos.png', dpi=120, bbox_inches='tight')
plt.show()

### O que reportar na competição

Para o Desafio Itaú, inclua no relatório:
1. **Turnover médio** da estratégia
2. **Premissa de custo** adotada e justificativa
3. **Métricas líquidas de custo** (não só brutas)
4. **Análise de sensibilidade**: "nossa estratégia mantém Sharpe > 1.0 até custos de X% por turno"

Isso mostra maturidade metodológica e diferencia da maioria dos grupos.

---
## 6. Walk-forward validation

A técnica mais honesta de avaliação de estratégias.

### Problema do backtest tradicional (in-sample)

```
Dataset completo: [──────────────────────────────────────]

Abordagem in-sample (ERRADA):
  Desenvolvimento:  [──────────────────────────────────────]
  Avaliação:        [──────────────────────────────────────]
  ↑ avaliando nos mesmos dados usados para construir
```

### Walk-forward (CORRETO)

```
Fold 1:  [TREINO──────────────] [TESTE───]
Fold 2:       [TREINO──────────────] [TESTE───]
Fold 3:            [TREINO──────────────] [TESTE───]
...                                             ↑
                                   Cada teste é sempre out-of-sample

Walk-forward result = concatenação de todos os períodos de TESTE
```

Para nosso caso: janela de treino = 48 meses, janela de teste = 12 meses.

In [ ]:
def walk_forward_momentum(ret_mensais, treino_meses=48, teste_meses=12, n_top=10):
    """
    Walk-forward validation para estratégia de momentum cross-sectional.
    
    Em cada fold:
    - Treino: calibra qualquer parâmetro livre (aqui só vol rolling, não temos params)
    - Teste: aplica a estratégia e registra o retorno out-of-sample
    
    Para nosso momentum simples (sem parâmetros a calibrar), o walk-forward
    serve principalmente para medir consistência ao longo do tempo.
    """
    datas = ret_mensais.index
    resultados_oos = []
    
    inicio = treino_meses
    while inicio + teste_meses <= len(datas):
        # Período de treino (para construir o sinal com os dados disponíveis)
        idx_treino_fim = inicio
        idx_teste_fim  = min(inicio + teste_meses, len(datas))
        
        # Período de teste
        datas_teste = datas[idx_treino_fim:idx_teste_fim]
        
        # Calcular sinal usando APENAS dados disponíveis até esse ponto
        ret_ate_agora = ret_mensais.iloc[:idx_treino_fim + teste_meses]
        sinal_wf = ret_ate_agora.shift(2).rolling(11).sum()
        
        for data in datas_teste:
            if data not in sinal_wf.index:
                continue
            sinal_hoje = sinal_wf.loc[:data].iloc[-2] if len(sinal_wf.loc[:data]) >= 2 else None
            if sinal_hoje is None:
                continue
            # Ranking e pesos
            rank = sinal_hoje.rank(ascending=True, pct=True)
            top  = (rank >= (1 - n_top / rank.notna().sum()))
            if top.sum() == 0:
                continue
            pesos_hoje = top.astype(float) / top.sum()
            # Retorno do mês seguinte (out-of-sample)
            loc_data = ret_mensais.index.get_loc(data)
            if loc_data + 1 >= len(ret_mensais):
                continue
            ret_amanha = ret_mensais.iloc[loc_data + 1]
            ret_oos = (pesos_hoje * ret_amanha).sum()
            resultados_oos.append({'data': data, 'retorno_oos': ret_oos})
        
        inicio += teste_meses
    
    return pd.DataFrame(resultados_oos).set_index('data')['retorno_oos']

print('Executando walk-forward (pode demorar ~30 segundos)...')
ret_wf = walk_forward_momentum(ret_mensais, treino_meses=48, teste_meses=12)
print(f'Walk-forward: {len(ret_wf)} meses out-of-sample')
print(f'Período: {ret_wf.index[0].strftime("%m/%Y")} a {ret_wf.index[-1].strftime("%m/%Y")}')

In [ ]:
# Comparar in-sample vs walk-forward (out-of-sample)
idx_comum = ret_v1.dropna().index.intersection(ret_wf.dropna().index)

m_is  = calcular_metricas(ret_v1.reindex(idx_comum).dropna(),
                           benchmark=ret_ibov.reindex(idx_comum),
                           nome='In-sample (v1)')
m_wf  = calcular_metricas(ret_wf.reindex(idx_comum).dropna(),
                           benchmark=ret_ibov.reindex(idx_comum),
                           nome='Walk-forward (OOS)')
m_ibov = calcular_metricas(ret_ibov.reindex(idx_comum).dropna(),
                            nome='IBOVESPA')

print('=== IN-SAMPLE vs WALK-FORWARD (OUT-OF-SAMPLE) ===')
exibir_metricas(m_is, m_wf, m_ibov)

degradacao_sharpe = (m_wf['sharpe'] / m_is['sharpe'] - 1)
print(f'\nDegradação do Sharpe (IS→OOS): {degradacao_sharpe:.1%}')
print('Degradação moderada é esperada e saudável.')
print('Degradação >70% indica overfitting severo.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Retorno cumulativo
ax = axes[0, 0]
((1 + ret_v1.reindex(idx_comum)).cumprod() - 1).plot(ax=ax, color='steelblue', lw=2, label='In-sample (v1)')
((1 + ret_wf.reindex(idx_comum)).cumprod() - 1).plot(ax=ax, color='orange', lw=2, label='Walk-forward (OOS)')
((1 + ret_ibov.reindex(idx_comum)).cumprod() - 1).plot(ax=ax, color='gray', lw=1.5, linestyle='--', label='IBOVESPA')
ax.set_title('Retorno cumulativo: IS vs OOS')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.legend()

# Retorno anual
ax = axes[0, 1]
ret_anual_is  = ret_v1.reindex(idx_comum).resample('YE').apply(lambda r: (1+r).prod()-1)
ret_anual_wf  = ret_wf.reindex(idx_comum).resample('YE').apply(lambda r: (1+r).prod()-1)
anos = ret_anual_is.index.year
x = np.arange(len(anos))
width = 0.35
ax.bar(x - width/2, ret_anual_is.values, width, label='In-sample', color='steelblue', alpha=0.8)
ax.bar(x + width/2, ret_anual_wf.values, width, label='Walk-forward', color='orange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(anos, rotation=45)
ax.axhline(0, color='black', lw=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.set_title('Retorno anual: IS vs OOS')
ax.legend()

# Rolling Sharpe 24 meses
ax = axes[1, 0]
rolling_is  = ret_v1.reindex(idx_comum).rolling(24).apply(lambda r: (r.mean()/r.std())*np.sqrt(12) if r.std()>0 else np.nan)
rolling_wf  = ret_wf.reindex(idx_comum).rolling(24).apply(lambda r: (r.mean()/r.std())*np.sqrt(12) if r.std()>0 else np.nan)
rolling_is.plot(ax=ax, color='steelblue', lw=1.5, label='In-sample', alpha=0.8)
rolling_wf.plot(ax=ax, color='orange', lw=1.5, label='Walk-forward', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.7)
ax.set_title('Rolling Sharpe 24 meses: consistência IS vs OOS')
ax.legend()

# Scatter IS vs OOS (retorno mensal)
ax = axes[1, 1]
idx_scatter = ret_v1.reindex(idx_comum).dropna().index.intersection(ret_wf.reindex(idx_comum).dropna().index)
x_vals = ret_v1.reindex(idx_scatter).values
y_vals = ret_wf.reindex(idx_scatter).values
ax.scatter(x_vals, y_vals, alpha=0.4, s=20, color='steelblue')
m_reg, b_reg, r_val, *_ = stats.linregress(x_vals, y_vals)
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
ax.plot(x_line, m_reg * x_line + b_reg, color='red', lw=2, label=f'OLS (R²={r_val**2:.2f})')
ax.axhline(0, color='black', lw=0.5)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Retorno mensal in-sample')
ax.set_ylabel('Retorno mensal walk-forward (OOS)')
ax.set_title('Correlação IS × OOS: os retornos se alinham?')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.legend()

plt.suptitle('Walk-forward Validation: In-sample vs Out-of-sample', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('walkforward_validation.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Resumo: hierarquia de evidências

Reunindo tudo que vimos, existe uma **hierarquia de confiança** nos resultados de um backtest:

| Nível | Tipo de resultado | Confiança | Nossa estratégia |
|-------|------------------|-----------|------------------|
| ① | In-sample, sem custo, parâmetro otimizado | Mínima | — |
| ② | In-sample, sem custo, parâmetro fundamentado | Baixa | Aulas 4-7 |
| ③ | In-sample, **com custo**, parâmetro fundamentado | Média | Esta aula |
| ④ | **Walk-forward**, com custo, parâmetro fundamentado | Alta | Esta aula |
| ⑤ | Walk-forward + live trading paper | Muito alta | Além do scope |

Para o Desafio Itaú, **nível ④** é o mais honesto e diferenciado.

In [ ]:
# Tabela final: todos os cenários lado a lado
custo_medio = 0.005  # 0.5% por turno
custo_mensal_wf = (turnover_v2 * custo_medio).reindex(ret_wf.index).fillna(0)
ret_wf_liquido  = ret_wf - custo_mensal_wf

custo_mensal_is = (turnover_v2 * custo_medio).reindex(ret_v2.index).fillna(0)
ret_v2_liquido  = ret_v2 - custo_mensal_is

idx_final = ret_wf_liquido.dropna().index.intersection(ret_v2_liquido.dropna().index)

m_naive    = calcular_metricas(ret_v1.reindex(idx_final).dropna(),
                                benchmark=ret_ibov.reindex(idx_final), nome='① IS bruto (v1)')
m_is_liq   = calcular_metricas(ret_v2_liquido.reindex(idx_final).dropna(),
                                benchmark=ret_ibov.reindex(idx_final), nome='③ IS líquido (v2)')
m_wf_liq   = calcular_metricas(ret_wf_liquido.reindex(idx_final).dropna(),
                                benchmark=ret_ibov.reindex(idx_final), nome='④ WF líquido (v2)')
m_bench    = calcular_metricas(ret_ibov.reindex(idx_final).dropna(), nome='IBOVESPA')

print('=== COMPARAÇÃO FINAL: TODOS OS CENÁRIOS ===')
exibir_metricas(m_naive, m_is_liq, m_wf_liq, m_bench)

print('\n→ O resultado walk-forward líquido de custos é o número que conta.')
print('→ Diferença entre ① e ④ = ilusão de backtest.')

In [ ]:
# Salvar retorno walk-forward líquido para uso na Aula 9 e 10
ret_wf_liquido.to_frame(name='retorno').to_parquet(os.path.join(DADOS_DIR, 'retorno_walkforward_liquido.parquet'))

print('Arquivo salvo: retorno_walkforward_liquido.parquet')
print()
print('=== CHECKLIST DO BACKTEST RIGOROSO ===')
checklist = [
    ('Look-ahead bias', 'shift(2) correto no sinal, pesos deslocados com shift(1)', True),
    ('Survivorship bias', 'Usando IBOV atual (parcial — risco residual)', False),
    ('Parâmetro fundamentado', 'Janela 11m baseada em Jegadeesh & Titman (1993)', True),
    ('Custos de transação', f'0.5% por turno × turnover médio {turnover_v2.mean():.1%}', True),
    ('Walk-forward OOS', 'Treino 48m, teste 12m, rolling', True),
    ('Deflated Sharpe', 'Fundamento econômico → ~1 teste efetivo', True),
    ('Transaction costs break-even', 'Análise de sensibilidade incluída', True),
]
for item, detalhe, ok in checklist:
    status = '✓' if ok else '~'
    print(f'  [{status}] {item}: {detalhe}')

---
## Conclusão da Aula 8

Um backtest rigoroso não é mais pessimista — é mais **honesto**. E um resultado honesto que ainda mostra alpha é muito mais valioso do que um resultado inflado que falha ao vivo.

### O que aprendemos

1. **Look-ahead bias** infla Sharpe artificialmente — `.shift()` correto é a proteção
2. **Overfitting** acontece quando otimizamos parâmetros sem fundamento — teoria vem antes dos dados
3. **Multiple testing** faz qualquer estratégia parecer boa se testamos o suficiente — precisamos corrigir pelo número de testes
4. **Deflated Sharpe** quantifica a barreira estatística considerando os testes realizados
5. **Custos de transação** são reais e precisam estar no relatório — com análise de sensibilidade
6. **Walk-forward** é a forma mais honesta: sempre testamos no futuro, nunca no passado

### Para o relatório da competição

Mencione explicitamente:
- Que você validou ausência de look-ahead bias
- Que os parâmetros foram escolhidos por fundamento econômico (cite J&T 1993)
- Turnover médio e premissa de custo adotada
- Métricas walk-forward alongside métricas in-sample

---
### Próxima aula

**Aula 9 — GenAI + Análise:** como usar LLMs para escrever as seções analíticas do relatório — e onde eles ajudam (e onde atrapalham) no processo quant.